# Loop0, stage 03: background cutoffs and logicle parameters

For each antibody this measures where real staining begins, using the control
that leaves that antibody out, and turns that into the parameters of the logicle
transformation.

The control cells alone are not enough.  The logicle transformation also needs
the top of the display range, which comes from the stained cells of the screen.
Both are concatenated here, the controls labelled by the antibody they leave out
and the screen labelled `fully_stained`.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Loop0_FMOs_Gated.h5ad` | stage 02b | the gated control cells |
| `Loop0_Gated_ExpRem.h5ad` | stage 02 | the gated cells of the screen |

## Outputs

| file | holds |
|---|---|
| `marker_cutoff.pkl` | one background cutoff per antibody |
| `logicle_parameters.json` | the four logicle parameter lists `t`, `m`, `w` and `a` |

Both outputs are used by stage 04 of every loop, not only this one.  The other
loops carry a copy of these two files rather than recomputing them.

Run the cells from top to bottom.

In [ ]:
import anndata as ann
import matplotlib.pyplot as plt
import scanpy as sc

from labcompass import (
    compute_marker_cutoffs,
    save_logicle_parameters,
    save_marker_cutoffs,
)

IN_FMO_GATED = "Loop0_FMOs_Gated.h5ad"
IN_SCREEN = "Loop0_Gated_ExpRem.h5ad"

OUT_CUTOFFS = "marker_cutoff.pkl"
OUT_LOGICLE = "logicle_parameters.json"

# Everything above this percentile of a control channel is treated as outlier
# noise rather than background, and dropped before the maximum is taken.
FMO_PERCENTILE = 99.5
# The same idea for the stained cells, which set the top of the display range.
STAINED_PERCENTILE = 99.75

In [ ]:
adata = sc.read(IN_FMO_GATED)
antibody_list = adata.var["antibody"].unique().tolist()
print(f"{len(antibody_list)} antibodies:", antibody_list)

## Add the stained cells of the screen

Their `fmo` label is `fully_stained`, which is how `compute_marker_cutoffs` tells
them apart from the controls.

In [ ]:
screen = sc.read(IN_SCREEN)
screen.obs["fmo"] = "fully_stained"

adata = ann.concat(
    [adata, screen], join="inner", merge="same", uns_merge="unique",
    label="batch", index_unique="-",
)
del screen
adata.obs["fmo"].value_counts()

## Measure the background

Each panel shows one antibody.  The red histogram is the control that leaves that
antibody out, so it is background.  The black histogram is the stained cells.
The title gives the highest background intensity, which becomes that antibody's
cutoff.

In [ ]:
marker_cutoff, t, m, w, a = compute_marker_cutoffs(
    adata,
    antibody_list,
    fmo_percentile=FMO_PERCENTILE,
    stained_percentile=STAINED_PERCENTILE,
    plot=True,
)
plt.show()
marker_cutoff

## Write the two calibration files

In [ ]:
save_marker_cutoffs(marker_cutoff, OUT_CUTOFFS)
save_logicle_parameters(t, m, w, a, OUT_LOGICLE)
print(f"wrote {OUT_CUTOFFS} with {len(marker_cutoff)} antibodies")
print(f"wrote {OUT_LOGICLE} with {len(t)} channels")